# Audting the data for changes in bhavcopy formats over the years

In [1]:
from datetime import date, timedelta
import time
import pandas as pd
from jugaad_data.nse import bhavcopy_save

RAW = "/Users/manishkumar/quant/data/raw"

In [3]:
dates = pd.date_range('2008-01-01', '2026-09-01', freq = 'MS') + pd.Timedelta(days = 14) 

# Initial loop that fetches the format of a bhavcopy instance (1 day randomly permonth) and adds the format to results

In [5]:
results = []
for d in dates:
    d = d.date()
    while d.weekday() >= 5:
        d += timedelta(days = 1)
    try:
        path = bhavcopy_save(d, RAW)
        cols = tuple(pd.read_csv(path, nrows = 0).columns)
        results.append((d, cols))
    except Exception as e:
        print(d, "failed:", e)
    time.sleep(1)

2008-08-15 failed: 404 Client Error: Not Found for url: https://www.nseindia.com/api/reports?archives=%5B%7B%22name%22%3A+%22CM+-+Bhavcopy%28csv%29%22%2C+%22type%22%3A+%22daily-reports%22%2C+%22category%22%3A+%22capital-market%22%2C+%22section%22%3A+%22equities%22%7D%5D&date=15-Aug-2008&type=equities&mode=single
2008-11-17 failed: 404 Client Error: Not Found for url: https://www.nseindia.com/api/reports?archives=%5B%7B%22name%22%3A+%22CM+-+Bhavcopy%28csv%29%22%2C+%22type%22%3A+%22daily-reports%22%2C+%22category%22%3A+%22capital-market%22%2C+%22section%22%3A+%22equities%22%7D%5D&date=17-Nov-2008&type=equities&mode=single
2009-08-17 failed: 404 Client Error: Not Found for url: https://www.nseindia.com/api/reports?archives=%5B%7B%22name%22%3A+%22CM+-+Bhavcopy%28csv%29%22%2C+%22type%22%3A+%22daily-reports%22%2C+%22category%22%3A+%22capital-market%22%2C+%22section%22%3A+%22equities%22%7D%5D&date=17-Aug-2009&type=equities&mode=single
2009-10-15 failed: 404 Client Error: Not Found for url: ht

# A sanity check here to detect whether some missed dates were errors in the moment

In [58]:
from jugaad_data.nse import stock_df

check = stock_df('RELIANCE', from_date=date(2021, 6, 14), to_date=date(2021, 6, 16), series='EQ')
check['DATE']

/opt/homebrew/Caskroom/miniforge/base/envs/quant_jvan/lib/python3.10/site-packages/jugaad_data/util.py:40: UserWarning: no explicit representation of timezones available for np.datetime64
  return np.datetime64(dt)


0   2021-06-15 18:30:00
1   2021-06-14 18:30:00
2   2021-06-13 18:30:00
Name: DATE, dtype: datetime64[ns]

# Looping over missed dates and trying again

In [62]:
done  = {d for d, c in results}
failed = []
for d in dates:
    d = d.date()
    while d.weekday() >= 5:
        d += timedelta(days=1)
    if d not in done:
        failed.append(d)
len(failed)

23

In [19]:
still_failed = []
for d in failed:
    try:
        path = bhavcopy_save(d, RAW)
        cols = tuple(pd.read_csv(path, nrows = 0).columns)
        results.append((d, cols))
    except Exception as e:
        still_failed.append(d)
    time.sleep(1)

# Audit here finds out different formats of bhavcopy over the years and collect some statistics

In [57]:
audit = pd.DataFrame(results, columns=['date', 'cols'])
audit.groupby('cols')['date'].agg(['min', 'max', 'count'])

,min,max,count
cols,,,
"(SYMBOL, SERIES, OPEN, HIGH, LOW, CLOSE, LAST, PREVCLOSE, TOTTRDQTY, TOTTRDVAL, TIMESTAMP, TOTALTRADES, ISIN, Unnamed: 13)",2011-07-15,2024-05-15,153
"(SYMBOL, SERIES, OPEN, HIGH, LOW, CLOSE, LAST, PREVCLOSE, TOTTRDQTY, TOTTRDVAL, TIMESTAMP, Unnamed: 11)",2008-01-15,2011-06-15,47
"(TradDt, BizDt, Sgmt, Src, FinInstrmTp, FinInstrmId, ISIN, TckrSymb, SctySrs, XpryDt, FininstrmActlXpryDt, StrkPric, OptnTp, FinInstrmNm, OpnPric, HghPric, LwPric, ClsPric, LastPric, PrvsClsgPric, UndrlygPric, SttlmPric, OpnIntrst, ChngInOpnIntrst, TtlTradgVol, TtlTrfVal, TtlNbOfTxsExctd, SsnId, NewBrdLotQty, Rmks, Rsvd1, Rsvd2, Rsvd3, Rsvd4)",2024-07-15,2026-09-15,23


In [50]:
summary = audit.groupby('cols')['date'].agg(['min', 'max', 'count'])
summary = summary.sort_values('min')
f1, f2, f3 = summary.index

In [51]:
audit.shape

(223, 3)

In [55]:
dates_3 = audit.groupby('cols').get_group(f3)['date'].sort_values()

In [56]:
for d in dates_3:
    df = pd.read_csv(bhavcopy_save(d, RAW))
    same = (df['TradDt'] == df['BizDt']).all()
    print(d, same)

2024-07-15 True
2024-09-16 True
2024-10-15 True
2024-12-16 True
2025-01-15 True
2025-02-17 True
2025-03-17 True
2025-04-15 True
2025-05-15 True
2025-06-16 True
2025-07-15 True
2025-09-15 True
2025-10-15 True
2025-11-17 True
2025-12-15 True
2026-02-16 True
2026-03-16 True
2026-04-15 True
2026-05-15 True
2026-06-15 True
2026-07-15 True
2026-08-17 True
2026-09-15 True
